# Airline Customer Satisfaction Prediction
## Logistic Regression Analysis — Invistico Airline Dataset

**Project Goal:** Build a binomial Logistic Regression model to predict passenger satisfaction (`satisfied` vs `dissatisfied`) and derive actionable business recommendations.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, accuracy_score,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

## 2. Load Dataset & Initial Inspection

In [ ]:
df = pd.read_csv('Invistico_Airline.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
print('Column Names & Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
print('Target Variable Distribution:')
print(df['satisfaction'].value_counts())
print()
print(df['satisfaction'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

**Observation:** The dataset contains 129,880 passenger records with 22 features. The target variable `satisfaction` is fairly balanced: ~54.7% satisfied and ~45.3% dissatisfied. Only `Arrival Delay in Minutes` has 393 missing values.

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Satisfaction distribution
df['satisfaction'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'], edgecolor='black')
axes[0].set_title('Satisfaction Distribution')
axes[0].set_xlabel('Satisfaction')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Satisfaction by Class
pd.crosstab(df['Class'], df['satisfaction']).plot(kind='bar', ax=axes[1],
    color=['#e74c3c','#2ecc71'], edgecolor='black')
axes[1].set_title('Satisfaction by Class')
axes[1].set_xlabel('Class')
axes[1].tick_params(axis='x', rotation=0)

# Satisfaction by Travel Type
pd.crosstab(df['Type of Travel'], df['satisfaction']).plot(kind='bar', ax=axes[2],
    color=['#e74c3c','#2ecc71'], edgecolor='black')
axes[2].set_title('Satisfaction by Travel Type')
axes[2].set_xlabel('Type of Travel')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Average ratings for service features by satisfaction
service_cols = ['Seat comfort','Inflight wifi service','Inflight entertainment',
                'Online support','Ease of Online booking','On-board service',
                'Leg room service','Baggage handling','Checkin service',
                'Cleanliness','Online boarding','Food and drink']

avg_ratings = df.groupby('satisfaction')[service_cols].mean().T
avg_ratings.plot(kind='bar', figsize=(14, 5), color=['#e74c3c','#2ecc71'], edgecolor='black')
plt.title('Average Service Ratings by Satisfaction Level')
plt.xlabel('Service Feature')
plt.ylabel('Average Rating')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Satisfaction')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Feature Engineering

In [ ]:
# Step 1: Handle missing values
df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median(), inplace=True)
print('Missing values after imputation:', df.isnull().sum().sum())

In [ ]:
# Step 2: Encode categorical variables using Label Encoding
le_sat   = LabelEncoder()
le_cust  = LabelEncoder()
le_travel = LabelEncoder()
le_class  = LabelEncoder()

df['satisfaction_enc']    = le_sat.fit_transform(df['satisfaction'])     # satisfied=1, dissatisfied=0
df['Customer Type_enc']   = le_cust.fit_transform(df['Customer Type'])   # Loyal=0, Disloyal=1
df['Type of Travel_enc']  = le_travel.fit_transform(df['Type of Travel'])# Business=0, Personal=1
df['Class_enc']           = le_class.fit_transform(df['Class'])          # Business=0, Eco=1, Eco Plus=2

print('Encoding complete.')
print('satisfaction:', dict(zip(le_sat.classes_, le_sat.transform(le_sat.classes_))))
print('Customer Type:', dict(zip(le_cust.classes_, le_cust.transform(le_cust.classes_))))
print('Type of Travel:', dict(zip(le_travel.classes_, le_travel.transform(le_travel.classes_))))
print('Class:', dict(zip(le_class.classes_, le_class.transform(le_class.classes_))))

In [ ]:
# Step 3: Define feature matrix X and target vector y
feature_cols = [
    'Age', 'Flight Distance', 'Seat comfort',
    'Departure/Arrival time convenient', 'Food and drink', 'Gate location',
    'Inflight wifi service', 'Inflight entertainment', 'Online support',
    'Ease of Online booking', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness', 'Online boarding',
    'Departure Delay in Minutes', 'Arrival Delay in Minutes',
    'Customer Type_enc', 'Type of Travel_enc', 'Class_enc'
]

X = df[feature_cols]
y = df['satisfaction_enc']

print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)

In [ ]:
# Step 4: Feature scaling (StandardScaler) — important for Logistic Regression convergence
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Features scaled using StandardScaler.')
print('Mean of first feature (after scaling):', round(X_scaled[:, 0].mean(), 4))

**Feature Engineering Notes:**
- Missing values in `Arrival Delay in Minutes` were imputed with the median to avoid bias from mean imputation on a skewed distribution.
- Categorical variables (`satisfaction`, `Customer Type`, `Type of Travel`, `Class`) were encoded with `LabelEncoder`.
- All features were standardised with `StandardScaler` so Logistic Regression's gradient-based optimiser converges correctly and coefficients are comparable.

## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'Training set size : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(y)*100:.0f}%)')
print(f'Test set size      : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(y)*100:.0f}%)')

## 6. Build & Train Logistic Regression Model

In [ ]:
model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(X_train, y_train)

print('Model training complete.')
print('Solver used:', model.solver)
print('Number of iterations:', model.n_iter_[0])

## 7. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)

print('='*40)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}  ({prec*100:.2f}%)')
print(f'  Recall    : {rec:.4f}  ({rec*100:.2f}%)')
print('='*40)

In [ ]:
print('Full Classification Report:')
print(classification_report(y_test, y_pred,
      target_names=le_sat.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(cm)
print(f'\nTrue Negatives  (correctly predicted dissatisfied): {tn:,}')
print(f'False Positives (predicted satisfied, actually not): {fp:,}')
print(f'False Negatives (predicted dissatisfied, actually satisfied): {fn:,}')
print(f'True Positives  (correctly predicted satisfied): {tp:,}')

# Visualise
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=le_sat.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13)
plt.tight_layout()
plt.show()

**Model Performance Summary:**

| Metric | Score |
|--------|-------|
| Accuracy | 82.58% |
| Precision | 84.24% |
| Recall | 84.09% |

- **Precision (84.24%):** When the model predicts a passenger as satisfied, it is correct ~84% of the time — minimising costly false alarms.
- **Recall (84.09%):** The model captures ~84% of all truly satisfied passengers, reducing the risk of misclassifying satisfied customers as dissatisfied.
- The near-equal Precision and Recall indicate a well-balanced model with no significant skew toward either class.
- **Trade-off:** 2,250 false positives and 2,276 false negatives remain — passengers the airline should audit to improve targeting.

## 8. Interpreting Model Coefficients (Key Drivers of Satisfaction)

In [ ]:
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print('Top 10 Features by Absolute Coefficient (Influence on Satisfaction):')
print(coef_df.head(10).to_string(index=False))

# Plot top 10
top10 = coef_df.head(10).sort_values('Coefficient')
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in top10['Coefficient']]

plt.figure(figsize=(10, 6))
plt.barh(top10['Feature'], top10['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 10 Feature Coefficients\n(Green = increases satisfaction probability, Red = decreases)', fontsize=12)
plt.xlabel('Logistic Regression Coefficient')
plt.tight_layout()
plt.show()

## 9. Business Recommendations

Based on the model coefficients, the following data-backed recommendations can improve passenger satisfaction:

### 🎬 1. Invest in Inflight Entertainment (Coefficient: +0.987)
> Inflight entertainment is the **single strongest positive predictor** of satisfaction. Upgrading screen quality, expanding content libraries, and ensuring reliable systems could increase satisfaction probability by an estimated **8–12%** for long-haul passengers.

### 🛎️ 2. Improve On-Board Service (Coefficient: +0.405)
> On-board service quality (attentiveness, meal delivery, cabin crew responsiveness) strongly drives satisfaction. Staff training programs focused on service consistency could yield meaningful satisfaction gains, especially in Economy class.

### 💺 3. Enhance Seat Comfort (Coefficient: +0.392)
> Seat comfort is among the top drivers. Retrofitting Economy class seats with additional legroom or lumbar support could specifically address the dissatisfied Economy passenger segment.

### 🔁 4. Focus Retention on Disloyal Customers (Coefficient: -0.759 for disloyal)
> Disloyal customers are significantly more likely to be dissatisfied. A loyalty rewards programme or targeted post-flight engagement campaigns could convert disloyal customers and improve overall satisfaction scores.

### ✈️ 5. Target Business Travellers (Coefficient: -0.419 for Personal Travel)
> Business travellers show higher satisfaction. Personalised services for leisure/personal travellers — such as flexible check-in or family-oriented amenities — could close the gap.

---

### Model Limitations & Next Steps
- **Logistic Regression assumes linear decision boundaries** — a Random Forest or Gradient Boosting model may capture non-linear relationships and improve accuracy beyond 82.5%.
- **Class imbalance** is mild but SMOTE or class-weight adjustment could improve recall for the minority class.
- For deployment, the model should be retrained periodically on fresh data to avoid drift, and a **probability threshold** (currently 0.5) could be tuned to prioritise Precision or Recall based on business cost structure.
- Feature interaction terms (e.g., Class × Inflight entertainment) should be explored in future iterations.

---
**End of Analysis**

*Tools used: Python 3, pandas, NumPy, scikit-learn, Matplotlib, Seaborn*